In [1]:
from prompt import SYSTEM_PROMPT

In [3]:
import os
import json
import uuid
import threading
import tempfile
import io

from flask import Flask, request, jsonify, render_template, send_file
from auditengine_new import run_llm_audit, highlight_pdf


BASE_DIR = (
    os.path.abspath(os.path.dirname(__file__))
    if "__file__" in globals()
    else os.getcwd()
)

DATA_DIR = os.path.join(BASE_DIR, "data")

UPLOAD_DIR = os.path.join(DATA_DIR, "uploads")
OUTPUT_DIR = os.path.join(DATA_DIR, "outputs")
STATUS_DIR = os.path.join(DATA_DIR, "status")
REFERENCE_DIR = os.path.join(DATA_DIR, "reference")

for d in [UPLOAD_DIR, OUTPUT_DIR, STATUS_DIR, REFERENCE_DIR]:
    os.makedirs(d, exist_ok=True)


app = Flask(__name__)


# ===============================
# Status helpers (local JSON)
# ===============================
def save_status(run_id, status, extra=None):
    data = {"status": status}
    if extra:
        data.update(extra)

    status_path = os.path.join(STATUS_DIR, f"{run_id}.json")
    with open(status_path, "w", encoding="utf-8") as f:
        json.dump(data, f)


def get_status(run_id):
    status_path = os.path.join(STATUS_DIR, f"{run_id}.json")
    if not os.path.exists(status_path):
        return None

    with open(status_path, "r", encoding="utf-8") as f:
        return json.load(f)


# ===============================
# Routes
# ===============================
@app.route("/")
def index():
    return render_template("index.html")


@app.route("/upload", methods=["POST"])
def upload():
    file = request.files.get("pdf")
    if not file or not file.filename.lower().endswith(".pdf"):
        return jsonify({"error": "Only PDF files allowed"}), 400

    run_id = str(uuid.uuid4())
    local_path = os.path.join(UPLOAD_DIR, f"{run_id}.pdf")

    file.save(local_path)
    save_status(run_id, "uploaded")

    return jsonify({"run_id": run_id})


@app.route("/run-audit/<run_id>", methods=["POST"])
def run_audit(run_id):

    def process():
        try:
            save_status(run_id, "processing")

            # ---- Input
            input_pdf = os.path.join(UPLOAD_DIR, f"{run_id}.pdf")

            # ---- Reference docs (must exist locally)
            gt = os.path.join(REFERENCE_DIR, "RBI-KFS.pdf")
            clm = os.path.join(REFERENCE_DIR, "CLM Guidelines1.pdf")
            gl = os.path.join(REFERENCE_DIR, "New-Gold-Loan-Regulations.pdf")

            output_pdf = os.path.join(OUTPUT_DIR, f"{run_id}.pdf")
            output_excel = os.path.join(OUTPUT_DIR, f"{run_id}.xlsx")

            results = run_llm_audit(
                ground_truth=gt,
                clm=clm,
                GL_regulation=gl,
                target_doc=input_pdf,
                user_prompt="",
                output_excel_path=output_excel
            )

            # ---- Not a loan document case
            if (
                isinstance(results, list)
                and len(results) == 1
                and results[0].get("whats_wrong") == "Not a loan document"
            ):
                save_status(
                    run_id,
                    "not_loan",
                    {"message": "This is not a loan document"}
                )
                return

            # ---- Highlight PDF
            highlight_pdf(input_pdf, output_pdf, results)

            save_status(
                run_id,
                "completed",
                {
                    "pdf": output_pdf,
                    "excel": output_excel
                }
            )

        except Exception as e:
            save_status(run_id, "failed", {"error": str(e)})

    threading.Thread(target=process, daemon=True).start()
    return jsonify({"status": "started"})


@app.route("/status/<run_id>")
def status(run_id):
    data = get_status(run_id)
    if not data:
        return jsonify({"error": "Invalid run ID"}), 404

    if data["status"] == "not_loan":
        return jsonify(data)

    if data["status"] == "completed":
        return jsonify({
            "status": "completed",
            "pdf": f"/download/{run_id}/pdf",
            "excel": f"/download/{run_id}/excel"
        })

    return jsonify(data)


@app.route("/download/<run_id>/<file_type>")
def download_file(run_id, file_type):
    data = get_status(run_id)
    if not data or data["status"] != "completed":
        return "File not ready", 404

    if file_type == "pdf":
        path = data["pdf"]
        mimetype = "application/pdf"
        filename = "audit_report.pdf"
    elif file_type == "excel":
        path = data["excel"]
        mimetype = (
            "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet"
        )
        filename = "audit_report.xlsx"
    else:
        return "Invalid file type", 400

    return send_file(
        path,
        mimetype=mimetype,
        as_attachment=True,
        download_name=filename
    )


if __name__ == "__main__":
    app.run(host="0.0.0.0", port=8080)


 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:8080
 * Running on http://192.168.1.108:8080
Press CTRL+C to quit
127.0.0.1 - - [15/Jan/2026 23:07:21] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [15/Jan/2026 23:07:21] "GET /static/logo.png HTTP/1.1" 200 -
127.0.0.1 - - [15/Jan/2026 23:07:22] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [15/Jan/2026 23:07:32] "POST /upload HTTP/1.1" 200 -
127.0.0.1 - - [15/Jan/2026 23:07:35] "POST /run-audit/34c6cc93-01c8-48a7-a7fc-a9526b69498f HTTP/1.1" 200 -
127.0.0.1 - - [15/Jan/2026 23:07:38] "GET /status/34c6cc93-01c8-48a7-a7fc-a9526b69498f HTTP/1.1" 200 -
127.0.0.1 - - [15/Jan/2026 23:07:41] "GET /status/34c6cc93-01c8-48a7-a7fc-a9526b69498f HTTP/1.1" 200 -
127.0.0.1 - - [15/Jan/2026 23:07:44] "GET /status/34c6cc93-01c8-48a7-a7fc-a9526b69498f HTTP/1.1" 200 -
127.0.0.1 - - [15/Jan/2026 23:07:47] "GET /status/34c6cc93-01c8-48a7-a7fc-a9526b69498f HTTP/1.1" 200 -
127.0.0.1 - - [15/Jan/2026 23:07:50] "GET /status/34c6cc93-01c8-48a7-a7fc